In [ ]:
import torch

In [ ]:
class FXGraphCapture:
    def __init__(self):
        self.captured_graphs = []

    def capture_backend(self, gm, example_inputs):
        self.captured_graphs.append(gm)
        return gm.forward  # no compile
        # return inductor(gm, example_inputs)  # with compile

    def _get_dtype_and_shape(self, meta):
        val = meta.get("example_value", meta.get("val"))
        if val is not None and hasattr(val, "dtype") and hasattr(val, "shape"):
            return val.dtype, tuple(val.shape)
        return "-", "-"

    def print_captured_graphs(self):
        # Extract the graph
        for gm in self.captured_graphs:
            rows = [
                ("---", "---", "---", "---", "---"),
                ("NAME", "OP", "TARGET", "DTYPE", "SHAPE"),
                ("---", "---", "---", "---", "---")
            ]
            for node in gm.graph.nodes:
                dtype, shape = self._get_dtype_and_shape(node.meta)
                rows.append((node.name, node.op, str(node.target), str(dtype), str(shape)))
            w = [max(map(len, col)) for col in zip(*rows)]  # widths for each column
            for r in rows:
                print(f"{r[0]:{w[0]}} {r[1]:{w[1]}} {r[2]:{w[2]}} {r[3]:{w[3]}} {r[4]:{w[4]}}")

    def save_dot_graph(self, filename_no_ext):
        from torch.fx.passes.graph_drawer import FxGraphDrawer
        for i, gm in enumerate(self.captured_graphs):
            drawer = FxGraphDrawer(gm, f"small_model_{i}")
            dot = drawer.get_dot_graph()
            node_meta = {node.name: node.meta for node in gm.graph.nodes}
            for dot_node in dot.get_nodes():
                name = dot_node.get_name().strip('"')
                meta = node_meta.get(name)
                if not meta:
                    continue
                dtype, shape = self._get_dtype_and_shape(meta)
                if dtype == "-":
                    continue
                old_label = dot_node.get("label").strip('"')
                if old_label.startswith("{") and old_label.endswith("}"):
                    new_label = f"{old_label[:-1]}|dtype={dtype}|shape={shape}}}"
                else:
                    new_label = f"{old_label}\\n{dtype}\\n{shape}"
                new_label = "|".join([ll for ll in new_label.split("|") if "num_users" not in ll])  # remove 'num_users' from nodes
                new_label = new_label.replace("\\n", "").replace("\\l", "").replace("\n", "")       # remove random new lines bloating nodes
                dot_node.set("label", new_label)
            dot.write_svg(filename_no_ext+f"_{i}.svg")

In [ ]:
class MyMoE(torch.nn.Module):
    def __init__(self, C, E, K):
        super().__init__()
        self.K = K
        self.W_router = torch.nn.Parameter(torch.randn(C, E))
        self.W_expert_0 = torch.nn.Parameter(torch.randn(C*4, C))

    def forward(self, x):
        # This function is compiled to 4x (!) separate sub-graphs because of selector creating tensor with data-dependent shape
        B, T, C = x.shape
        K = self.K
        x_flat = x.reshape(-1, C)          # B*T, C
        logits = x_flat @ self.W_router    # B*T, E
        weights = torch.sigmoid(logits)    # B*T, E
        values, indices = torch.topk(weights, K, dim=-1)     # B*T, K
        x_flat_stacked = torch.stack([x_flat]*K, dim=1)      # B*T, K, C
        x_flat_stacked_flat = x_flat_stacked.reshape(-1, C)  # B*T*K, C
        indices_flat = indices.reshape(-1)                   # B*T*K
        indices_flat_sorted_indices = torch.argsort(indices_flat, stable=True)  # B*T*K
        x_flat_stacked_flat_sorted = x_flat_stacked_flat[indices_flat_sorted_indices]  # B*T*K, C
        
        # Dynamic slicing breaks the torch.compile, this makes one graph, but it's incomplete
        num_expert_0 = (indices==0).sum()   # 0D tensor
        out_flat_stacked_flat_sorted = torch.zeros(B*T*K, C*4)   # B*T*K, C*4
        out_flat_stacked_flat_sorted[0:num_expert_0] = x_flat_stacked_flat_sorted[0:num_expert_0] @ self.W_expert_0.t()
        out_flat_stacked_flat = torch.zeros(B*T*K, C*4)
        out_flat_stacked_flat[indices_flat_sorted_indices] = out_flat_stacked_flat_sorted   # B*T*K, C*4

        out_flat_stacked = out_flat_stacked_flat.reshape(B*T, K, C*4)
        out_flat_stacked_weighted = out_flat_stacked * values.unsqueeze(-1)
        out_flat = out_flat_stacked_weighted.sum(dim=1)   # B*T, C*4
        outputs = out_flat.reshape(B, T, C*4)
        return outputs

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k

fx_capture = FXGraphCapture()

moe = MyMoE(C=C, E=E, K=K)
moe = torch.compile(moe, backend=fx_capture.capture_backend)

x = torch.randn(B, T, C)
outputs = moe(x)
# loss = outputs.float().square().mean()
# loss.backward()

In [ ]:
outputs[0, :8, :6]

In [ ]:
fx_capture.print_captured_graphs()

In [ ]:
fx_capture.save_dot_graph('moe')

In [ ]:
print(outputs.shape)